[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doxav/astromodel_proving/blob/main/analysis/05_mechanistic_decomposition.ipynb)

In [ ]:
from pathlib import Path
import os, sys, subprocess


def _running_in_colab() -> bool:
    return "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules


if _running_in_colab():
    repo_url = os.environ.get("ASTROMODEL_REPO_URL", "https://github.com/doxav/astromodel_proving.git")
    repo_branch = os.environ.get("ASTROMODEL_REPO_BRANCH", "main")
    project_root = Path("/content") / "astromodel_proving"
    if not project_root.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", repo_branch, repo_url, str(project_root)], check=True)
    os.chdir(project_root)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    project_root = Path(os.environ.get("ASTROMODEL_PROJECT_ROOT", ".")).resolve()
    os.chdir(project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

# Step 05 — Mechanistic decomposition of accepted cell ensembles

This notebook validates the Step 05 reviewer-response pipeline. It loads accepted Step 04 cell-level ensembles, simulates hidden outputs with the canonical local astrocyte model, decomposes accepted fits into Kir/gap/leak mechanism summaries, clusters in effective/mechanism space, and records conservative claim scope.

The implementation reuses the local model and ATF conventions developed from `analysis/astro_atf_analysis_improved_sectioned.ipynb`. Step 05 can identify candidate mechanism regimes or compensation manifolds, but predictive and perturbation support remains pending until Step 06.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

from src.step05_mechanistic_decomposition import Step05Config, run_step05_mechanistic_decomposition

config = Step05Config(max_candidates=4, time_points=120, bootstrap_iterations=8, random_seed=13, write_outputs=True)
result = run_step05_mechanistic_decomposition(project_root, config)
out_dir = project_root / "outputs" / "mechanisms"
print(json.dumps(result["analysis_summary"], indent=2))

## 1. Accepted ensemble inventory

In [ ]:
clusters = result["mechanism_clusters"]
flux = result["accepted_fit_mechanisms"]
inventory = clusters.groupby(["region", "condition"], as_index=False).agg(
    n_cells=("file_id", "nunique"),
    n_candidates=("candidate_id", "nunique"),
    n_clusters=("mechanism_cluster", "nunique"),
)
inventory

## 2. Candidate-sweep flux decomposition table

In [ ]:
flux_cols = [
    "file_id", "region", "condition", "candidate_id", "sweep", "current_na", "simulation_status",
    "I_Kir_integral", "I_kgap_integral", "I_leak_integral", "gap_to_kir_integral_ratio",
    "gap_fraction", "kir_fraction", "leak_fraction", "K_o_peak", "K_o_recovery_error", "proxy_validity_class",
]
flux[flux_cols].head(12)

## 3. Mechanism-space clustering

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for label, sub in clusters.groupby("mechanism_cluster"):
    ax.scatter(sub["gap_fraction_mean"], sub["kir_fraction_mean"], s=80, label=label)
for _, row in clusters.iterrows():
    ax.annotate(str(row["candidate_id"]).split("__")[-1], (row["gap_fraction_mean"], row["kir_fraction_mean"]), fontsize=8)
ax.set_xlabel("Mean gap fraction")
ax.set_ylabel("Mean Kir fraction")
ax.set_title("Step 05 mechanism-space cluster view")
ax.legend(title="Cluster")
fig.tight_layout()
plt.show()

## 4. Flux decomposition for representative candidates

In [ ]:
reps = result["representatives"]
rep_flux = flux.merge(reps[["file_id", "candidate_id", "representative_rank"]], on=["file_id", "candidate_id"], how="inner")
fig, ax = plt.subplots(figsize=(7, 4))
for (rank, candidate_id), sub in rep_flux.groupby(["representative_rank", "candidate_id"]):
    sub = sub.sort_values("current_na")
    ax.plot(sub["current_na"], sub["gap_fraction"], marker="o", label=f"rep {rank} gap")
    ax.plot(sub["current_na"], sub["kir_fraction"], marker="s", linestyle="--", label=f"rep {rank} Kir")
ax.set_xlabel("Current (nA)")
ax.set_ylabel("Flux fraction")
ax.set_ylim(0, 1)
ax.set_title("Representative Kir/gap decomposition across sweeps")
ax.legend(ncol=2, fontsize=8)
fig.tight_layout()
plt.show()
reps[["representative_rank", "file_id", "condition", "mechanism_cluster", "dominant_mechanism", "claim_scope"]]

## 5. Compensation-vs-separated-mode interpolation diagnostics

In [ ]:
geometry = result["geometry_classification"]
geometry

## 6. Bootstrap stability table

In [ ]:
stability = result["bootstrap_cluster_stability"]
stability.describe(include="all")

## 7. DH/VH and condition occupancy table

In [ ]:
enrichment = result["region_mechanism_enrichment"]
enrichment.sort_values(["region", "condition", "mechanism_cluster"])

## 8. Explicit claim-scope table

In [ ]:
claims = result["claim_scope_table"]
claims

In [ ]:
assert (flux["simulation_status"] == "ok").any()
assert {"region", "condition", "mechanism_cluster"}.issubset(clusters.columns)
assert claims["forbidden_pre_step06_claim"].str.contains("candidate_degenerate_regimes").any()
print(f"Step 05 notebook validated and outputs saved under {out_dir}")